# Generalisation, or: the only number that matters

> Training accuracy is a lie you tell yourself. How to build a validation set that doesn't lie back, and what to do when the gap opens up.

Read this chapter at `/learn/06-generalisation/`. Exported from `src/content/chapters/06-generalisation.mdx` — edit there, not here.


Here's a model that gets 100% on its training data:

```
def predict(x):
    return lookup_table[x]
```

It's a dictionary. It has learned nothing, it will fail on the first new input it
sees, and it is *perfect* by the measure we've been using so far.

So we need a better measure. That's today.

I'll say up front: this is where most real projects fail, and the failures are
boring, repetitive, and completely avoidable once you've seen them once. Which is
what we're about to do.

## Overfitting, demonstrated

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline

rng = np.random.default_rng(3)
n = 22
x = np.sort(rng.uniform(0, 1, n))
y_true = lambda t: np.sin(2.2 * np.pi * t)
y = y_true(x) + rng.normal(0, 0.28, n)

plt.figure(figsize=(5, 3))
grid = np.linspace(0, 1, 300)
plt.plot(grid, y_true(grid), "k--", lw=1, label="truth")
plt.scatter(x, y, s=22, label="observed (noisy)")
plt.legend(); plt.tight_layout()

Twenty-two noisy dots, and a dashed line showing the truth they were generated
from. In real life you'd only have the dots.

Now let's fit polynomials of increasing degree. Degree 1 is a straight line.
Degree 18 can wiggle through very nearly anything you like.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(9.5, 2.9), sharey=True)
for ax, deg in zip(axes, [1, 4, 18]):
    model = make_pipeline(PolynomialFeatures(deg), LinearRegression())
    model.fit(x.reshape(-1, 1), y)
    ax.plot(grid, y_true(grid), "k--", lw=1)
    ax.plot(grid, model.predict(grid.reshape(-1, 1)), c="crimson")
    ax.scatter(x, y, s=16)
    ax.set_ylim(-2.2, 2.2); ax.set_title(f"degree {deg}")
plt.tight_layout()

Look at the three of them side by side.

Degree 1 **underfits**. It's too rigid to be a sine wave, and it's wrong in
roughly the same way everywhere. It has a firm opinion and the opinion is bad.

Degree 18 **overfits**. It threads through nearly every single point — including
the noise — and then between the points it does something completely unhinged.

Degree 4 is about right.

Now here's the uncomfortable part, and I want to state it plainly: **the degree-18
model has the lowest training error of the three.** If you picked your model by
training error, you'd pick the worst one. Every time.

In [ ]:
for deg in [1, 4, 9, 18]:
    m = make_pipeline(PolynomialFeatures(deg), LinearRegression()).fit(x.reshape(-1, 1), y)
    train = ((m.predict(x.reshape(-1, 1)) - y) ** 2).mean()
    true  = ((m.predict(grid.reshape(-1, 1)) - y_true(grid)) ** 2).mean()
    print(f"degree {deg:2d}   train MSE {train:8.4f}   true MSE {true:10.4f}")

Two columns, two completely different stories.

Training error falls, and keeps falling, monotonically. Error against the actual
underlying truth falls for a while — and then *explodes*.

Overfitting is memorising the noise.

The model has no way to tell which parts of your data are signal and which parts
are accident. Given enough flexibility it will fit both with equal enthusiasm —
and the noise, by definition, is not going to repeat.

## The validation set

In the cell above we cheated: we compared against `y_true`, which we only have
because we invented the data. In real life there is no `y_true` to check against.

So what *can* you do?

Something rather elegant, and once you see it you'll wonder why it took the field
until the 1930s to formalise. You can't get access to the truth — but you can
**hide some of your data from yourself**.

In [ ]:
from sklearn.model_selection import train_test_split

X = x.reshape(-1, 1)
X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.35, random_state=0)

print(f"{len(X_tr)} train, {len(X_va)} validation")
for deg in [1, 4, 9, 18]:
    m = make_pipeline(PolynomialFeatures(deg), LinearRegression()).fit(X_tr, y_tr)
    tr = ((m.predict(X_tr) - y_tr) ** 2).mean()
    va = ((m.predict(X_va) - y_va) ** 2).mean()
    print(f"degree {deg:2d}   train {tr:7.4f}   valid {va:10.4f}")

Compare that validation column to the "true MSE" column two cells up. Same shape.
Same story. Same warning at degree 18.

And we never once looked at the truth.

That's the whole trick, and I'd argue it's the most important methodological idea
in the entire field. Data you didn't train on is a stand-in for the future.

<div class="table-scroll">

| Split | Who touches it | What it's for |
|---|---|---|
| **Training** | the optimiser | fitting the parameters |
| **Validation** | you | choosing the model, the hyperparameters, when to stop |
| **Test** | nobody, until the very end | one honest estimate, once |

</div>

The test set exists because **you overfit too**.

Every time you look at a validation score and change something in response, you
leak a little information out of that set and into your model. Do it two hundred
times — which is a completely normal week — and your validation score has quietly
become optimistic by a few percent. You didn't do anything wrong. You just
looked, repeatedly, which is what looking does.

The test set is the defence, and it only works if you use it *once*.

A test set you've consulted five times isn't a test set. It's a second validation
set with a misleading name and an inflated sense of its own honesty.

## Splitting randomly is often wrong

`train_test_split` shuffles. That's correct when your rows are independent — and
they very frequently are not.

In [ ]:
from sklearn.linear_model import Ridge

t = np.arange(300)
series = np.cumsum(rng.normal(0, 1, 300)) + 0.05 * t     # a random walk with drift
feats  = np.column_stack([np.roll(series, k) for k in (1, 2, 3)])[5:]
target = series[5:]

# A: shuffled split — leaks the future into the past
Xa_tr, Xa_va, ya_tr, ya_va = train_test_split(feats, target, test_size=0.3, random_state=0)
a = Ridge().fit(Xa_tr, ya_tr).score(Xa_va, ya_va)

# B: honest split — train on the past, validate on the future
cut = int(len(feats) * 0.7)
b = Ridge().fit(feats[:cut], target[:cut]).score(feats[cut:], target[cut:])

print(f"random split  R^2 = {a:.3f}   <- flattering")
print(f"time-based    R^2 = {b:.3f}   <- what you would actually get")

The random split let the model train on Tuesday and Thursday, and then asked it
to predict Wednesday.

It will never, ever get to do that in production. Wednesday comes before
Thursday. You do not get to see Thursday first.

**Your split has to reproduce the structure of the real prediction task.** That
sentence is worth writing on something.

The rule generalises. Group your split by whatever unit generalisation has to
cross:

- **Time series** → split at a date. Always. No exceptions.
- **Multiple rows per patient / user / customer** → split by *person*, never by row.
  Otherwise the same person appears on both sides and the model recognises them.
- **Photographs from the same session** → split by session, for the same reason.
- **A Kaggle competition** → go and look at how *they* split it. It's a hint about
  what the real task is, and it's free.

Building the validation set is the most consequential decision in a project.

And it's made in the first hour, by someone who thinks it's boilerplate, while
they're still setting up. That mismatch — highest stakes, lowest attention — is
why this fails so often.

If your validation set doesn't resemble the deployment condition, every number
you produce afterwards is fiction. Well-constructed, precisely-formatted,
confidently-reported fiction, which is considerably worse than no number at all.

**"How big should the validation set be?"** Big enough that the score is stable.
With 200 validation examples, a single flipped prediction moves your accuracy by
0.5% — so don't chase 0.3% improvements. A rough rule: 20–30% for small datasets,
and a fixed few thousand for large ones.

**"What's `random_state=0` doing and do I need it?"** It fixes the shuffle so you
get the same split every run. Yes, use it. Without it you can't tell whether your
change helped or the split just moved.

**"Why does `train_test_split` return four things in that order?"** `X_train,
X_test, y_train, y_test` — features first, then targets. It catches everyone once.
If your accuracy is bizarrely near 50%, check you didn't swap them.

**"My validation score is better than my training score."** Something is wrong,
and it's worth chasing. Usual causes: your split is broken, you have augmentation
or dropout on during training and off during validation (which is normal and
explains a small gap), or your validation set happens to be much easier. It is
never good news.

## Cross-validation

With little data, a single split is noisy — you might just have drawn a lucky
35%. K-fold cross-validation splits the data $k$ ways, trains $k$ times, and
averages the results.

In [ ]:
from sklearn.model_selection import cross_val_score

for deg in [1, 4, 9, 18]:
    m = make_pipeline(PolynomialFeatures(deg), LinearRegression())
    scores = -cross_val_score(m, X, y, cv=5, scoring="neg_mean_squared_error")
    print(f"degree {deg:2d}   MSE {scores.mean():9.4f}  ± {scores.std():7.4f}")

Now look at that ± column, because it's the part people skip and it's the part
that matters.

It's telling you how much of the difference between two models is **real**, and
how much is just which rows happened to land where. Two models whose error bars
overlap are not distinguishable by this dataset. Choosing between them on the
mean alone isn't analysis — it's superstition with a decimal point.

Use cross-validation when data is small and training is cheap. Skip it when
training costs four GPU-hours. Nobody cross-validates a language model, and
nobody expects you to.

## Fighting overfitting

Five tools, roughly in the order you'll reach for them.

**1. More data.** Always the best answer when you can get it. Noise averages out;
signal doesn't. Boring, expensive, undefeated.

**2. A simpler model.** Fewer parameters, fewer features, a lower polynomial
degree, a shallower tree. Free, instant, and consistently underrated by people
who'd rather tune something.

**3. Regularisation.** Keep the flexible model, but charge it rent for using its
flexibility.

In [ ]:
from sklearn.linear_model import Ridge, Lasso
from sklearn.preprocessing import StandardScaler

for name, reg in [("none  ", LinearRegression()),
                  ("ridge ", Ridge(alpha=1e-3)),
                  ("lasso ", Lasso(alpha=1e-3, max_iter=50_000))]:
    m = make_pipeline(PolynomialFeatures(18), StandardScaler(), reg).fit(X_tr, y_tr)
    coefs = m[-1].coef_
    va = ((m.predict(X_va) - y_va) ** 2).mean()
    print(f"{name}  valid MSE {va:8.4f}   largest |coef| {np.abs(coefs).max():10.2f}   "
          f"nonzero {int((np.abs(coefs) > 1e-6).sum()):2d}/19")

Look at that "largest |coef|" column for the unregularised fit. Enormous numbers.

That's what overfitting looks like *numerically*, and I find it a genuinely
useful thing to have seen. To thread a wiggly curve through every point, the
polynomial needs huge positive and negative coefficients that very nearly cancel
each other out — a violently unstable balancing act that happens to pass through
your dots.

A norm penalty simply makes that balancing act expensive, and
the model stops attempting it.

Ridge (L2) shrinks every coefficient smoothly toward zero. Lasso (L1) drives many
of them to *exactly* zero, so it selects features as well as shrinking them —
look at the "nonzero" column.

The difference comes from geometry: L1's constraint region is a diamond, and a
diamond has corners sitting on the axes. Solutions land on corners. A corner on
an axis means that coefficient is exactly zero. It's a lovely little piece of
reasoning, and it's in the maths appendix if you'd like it drawn out.

**4. Early stopping.** Watch validation loss during training and stop when it
turns upward. It's regularisation that costs you nothing at all, and it's why
every training loop you'll ever see prints two numbers per epoch instead of one.

**5. Data augmentation.** Manufacture plausible variants — flip the image, crop
it, shift the colours. Ten thousand photos become effectively a hundred thousand,
and along the way you've taught the model something true: that a cat rotated five
degrees is still a cat. That's [Chapter 11](/learn/11-vision-and-transfer/).

Notice that every one of these is a deliberate **handicap**. You make the model
worse at the training data on purpose, in exchange for it being better at
everything else.

The instinct you already have for this is the one that says a type signature
should be as narrow as the job requires. A function generic over `T: Display`
where you only ever pass `&str` isn't more powerful in any useful sense — it's
just harder to reason about, and it lets in things you never wanted.

Regularisation is exactly that argument, applied to a fitted function instead of
a written one.

## The gap, and what it tells you

In [ ]:
baseline = ((y_va - y_tr.mean()) ** 2).mean()      # predict the mean, always

def diagnose(train_err, valid_err, baseline_err):
    # Is the model even beating "predict the mean" on its own training data?
    if train_err > 0.5 * baseline_err:
        return "UNDERFIT   — too rigid; add capacity or features"
    if valid_err > 3 * train_err:
        return "OVERFIT    — regularise, simplify, or get more data"
    return "REASONABLE — now go and improve the features"

print(f"baseline (predict the mean) MSE = {baseline:.3f}\n")
for deg in [1, 4, 18]:
    m = make_pipeline(PolynomialFeatures(deg), LinearRegression()).fit(X_tr, y_tr)
    tr = ((m.predict(X_tr) - y_tr) ** 2).mean()
    va = ((m.predict(X_va) - y_va) ** 2).mean()
    print(f"degree {deg:2d}  train {tr:7.4f}  valid {va:11.4f}  {diagnose(tr, va, baseline)}")

The two numbers *together* are a diagnosis. Either one alone is not.

- **Both bad** → underfitting. More capacity, more features, train longer.
- **Train good, valid bad** → overfitting. Use the list above.
- **Both good** → ship it. Then go and check for
  [leakage](/learn/03-the-shape-of-problems/), because "both good" is also
  exactly what leakage looks like from here.
- **Valid better than train** → your split is broken. Investigate. This is never
  good news, however nice it looks.

Four outcomes, four responses. Print both numbers every epoch and you'll know
which world you're in at all times.

Everything above is the textbook bias–variance picture. It is genuinely how small
and medium models behave, and it was the settled wisdom for about forty years.

It also does not survive contact with modern deep learning, and it's worth
knowing why before you read a paper that assumes you already do.

**Double descent.** Increase model capacity, and test error goes down, then up —
the classical overfitting curve you just plotted. But keep going. Past the point
where the model can interpolate the training set *exactly*, test error comes
**down again**, often to a better minimum than the classical sweet spot ever
reached.

This was documented systematically by Belkin and colleagues in 2019, and it means
"the model has more parameters than data points" is not, by itself, the alarm bell
it was taught as for four decades.

**The lottery-ticket view.** A large randomly-initialised network appears to
contain small subnetworks that happen to be well-positioned to train, and
training is partly a *search* over them. Under this view, over-parameterisation
isn't waste — it's what makes the search feasible. You buy a lot of tickets
because you only need one to win.

**Implicit regularisation.** Remember from chapter 5 that SGD doesn't find *any*
minimum of the training loss — it preferentially finds flat ones, and flat minima
generalise better. So the optimiser has been quietly doing regularisation that
nobody wrote down and nobody asked for.

None of this means the validation set stops mattering. It matters **more**,
precisely because the theory is now a less reliable guide and measurement is all
you've got.

What it does mean: when you read "this model has 8 billion parameters and 2
trillion training tokens, isn't that just overfitting?", the honest answer is more
interesting than yes.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.tree import DecisionTreeClassifier

data = load_breast_cancer()
Xc, yc = data.data, data.target
Xc_tr, Xc_va, yc_tr, yc_va = train_test_split(
    Xc, yc, test_size=0.3, random_state=0, stratify=yc)

# 1. Fit DecisionTreeClassifier at max_depth = 1, 3, 5, None.
#    Print training and validation accuracy for each. Where does the gap open?
#
# 2. What is the accuracy of always predicting the majority class?
#    Every number above must be read against it.
#
# 3. Use cross_val_score with cv=5 on the FULL dataset for the best depth.
#    Is the standard deviation larger or smaller than the differences
#    between your depths? What does that imply?

print("replace me")

For 2, `yc.mean()` gives you the fraction that are class 1 — the majority class
baseline is whichever of that and its complement is bigger.

For 3, the question is really: *if the fold-to-fold wobble is as big as the
difference between two models, have you actually learned which model is better?*

In [ ]:
for depth in [1, 3, 5, None]:
    m = DecisionTreeClassifier(max_depth=depth, random_state=0).fit(Xc_tr, yc_tr)
    tr, va = m.score(Xc_tr, yc_tr), m.score(Xc_va, yc_va)
    print(f"depth {str(depth):4s}  train {tr:.3f}  valid {va:.3f}   gap {tr - va:+.3f}")

print(f"\nmajority-class baseline: {max(yc.mean(), 1 - yc.mean()):.3f}")

best = DecisionTreeClassifier(max_depth=3, random_state=0)
s = cross_val_score(best, Xc, yc, cv=5)
print(f"5-fold at depth 3: {s.mean():.3f} ± {s.std():.3f}")
print("folds:", s.round(3))

Three things to carry away.

**The unconstrained tree hits 1.000 training accuracy.** It has memorised every
single row, which a tree can always do if you let it — while validation just
stalls. That's the gap in its purest, most undeniable form. It is our
`lookup_table` from the top of the page, wearing a nicer outfit.

**The baseline is 63%.** So a model sitting at 88% is genuinely doing something —
but it's rather less impressive than "88%" sounds when you say it in a meeting.
Always, always quote against the baseline.

**And the fold standard deviation is comparable to the gap between depth 3 and
depth 5.** Which means those two models are *not distinguishable* on this
dataset. Picking one over the other based on a single split is reading tea
leaves.

That last one is, I think, the most commonly committed statistical error in
applied machine learning. It happens daily, in production, and on every
leaderboard you've ever looked at. You now know how to not do it, which puts you
ahead of a startling number of people with the job title.

Tomorrow: the models you should actually reach for — most of which, you may be
relieved to hear, are not neural networks.